# LangGraph + Repository Intelligence Orchestrator (M20-aligned)

This notebook integrates the repository's **existing M20 Orchestrator** with LangGraph.

## Execution boundary

- LangGraph creates and validates the investigation plan, assigns `INVESTIGATOR` / `ANALYST` roles, and previews specialist routing.
- The repository's `Orchestrator.execute(...)` remains the owner of the task lifecycle: sequential dispatch, M19 request/result contracts, M20 task-budget propagation, retries, evidence/scope checks, report assembly, and final collaboration validation.
- The notebook passes `role_assignments=...`, so the Orchestrator does not assign roles a second time.
- The same `QueryRouter` is injected for actual Investigator execution. The preview uses a separate router instance so previewing does not mutate the execution router's routing memo.

**Scope:** this is deliberately not a LangGraph-owned per-task scheduler. It exercises the updated M20 orchestration through the repository's supported API. Moving task-level workflow control into LangGraph is a separate milestone.

## 1. Project path and imports

In [ ]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import replace
from pathlib import Path
from typing import Any, TypedDict

from IPython.display import display
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Prefer an explicit environment override; otherwise use the current directory.
# Do not hard-code a developer-specific Windows path into the notebook.
PROJECT_ROOT = Path(
    os.environ.get("REPO_INTELLIGENCE_ROOT", str(Path.cwd()))
).expanduser().resolve()
SRC_ROOT = PROJECT_ROOT / "src"

if not (SRC_ROOT / "repo_intelligence").is_dir():
    raise FileNotFoundError(
        "Could not find src/repo_intelligence under PROJECT_ROOT="
        f"{PROJECT_ROOT}. Set REPO_INTELLIGENCE_ROOT to the repository root."
    )

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from repo_intelligence.llm.config import load_env
from repo_intelligence.llm.openai_client import OpenAILLMClient
from repo_intelligence.planning import (
    InvestigationPlan,
    build_planning_request,
    load_architecture_context,
    plan_from_response,
    validate_plan,
)
from repo_intelligence.orchestration import (
    Orchestrator,
    RoleAssignment,
    assign_roles,
)
from repo_intelligence.multiagent import TaskPayload
from repo_intelligence.agent import JsonFileTraceSink
from repo_intelligence.repository_query import (
    M17_ORCHESTRATION_CONFIG,
    OFFLINE_COMMIT,
    QueryRouter,
    build_architect,
    build_engine,
    build_flow_investigator,
    build_memory_context,
    build_memory_manager,
    build_visualizer,
    default_artifacts_root,
    live_bindings,
    read_snapshot_metadata,
    resolve_read_target,
    verify_subject_snapshot,
)

load_env()
print(f"Project root: {PROJECT_ROOT}")

## 2. Resolve and verify the pinned subject repository

In [ ]:
repository_root, readable_target, resolve_error = resolve_read_target()

if resolve_error:
    raise RuntimeError(f"Could not resolve subject repository: {resolve_error}")

verified_commit, commit_source, verification_error = verify_subject_snapshot(
    repository_root
)
if verification_error:
    raise RuntimeError(
        "Subject snapshot verification failed; refusing to analyze it: "
        f"{verification_error}"
    )

snapshot_metadata = read_snapshot_metadata(repository_root)
repository_label = (
    snapshot_metadata.get("name")
    or repository_root.name
    or str(repository_root)
)
current_commit = verified_commit or OFFLINE_COMMIT

print(f"Repository root: {repository_root}")
print(f"Repository scope: {repository_label}")
print(f"Commit scope: {current_commit}")
print(f"Commit source: {commit_source}")
print(f"Readable entry target: {readable_target}")

## 3. Load Architecture Context and initialize shared runtime

In [ ]:
# The planner consumes the already-generated Architecture Context artifact.
architecture_context, architecture_context_record = load_architecture_context(
    output_root=default_artifacts_root()
)

# Keep the repository's existing MemoryManager / MemoryContext / ContextEngine
# and tool-binding construction. The notebook does not invent parallel memory.
memory_manager = build_memory_manager()
memory_context = build_memory_context(
    repository=repository_label,
    current_commit=current_commit,
)
context_engine = build_engine(repository_label)
trace_sink = JsonFileTraceSink()

tool_bindings = live_bindings(
    repository_root,
    output_root=default_artifacts_root(),
)

print("Architecture Context:", architecture_context_record)
print("Shared runtime initialized.")

## 4. Build the existing specialists, QueryRouter, and M20 Orchestrator

In [ ]:
# Separate LLM clients preserve the repository's specialist responsibilities.
planner_llm = OpenAILLMClient()
architect_llm = OpenAILLMClient()
flow_llm = OpenAILLMClient()
visualizer_llm = OpenAILLMClient()

architect = build_architect(
    llm=architect_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

flow_investigator = build_flow_investigator(
    llm=flow_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

visualizer = build_visualizer(
    llm=visualizer_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
)

# Actual Investigator dispatch uses this router.
router = QueryRouter(
    architecture=architect,
    flow=flow_investigator,
)

# M20: activate the real per-attempt task budget. The executor passes this
# limit to run_result(..., max_tool_calls=...), where the agent's pre-tool
# guard enforces it. Change the environment variable to tune the limit.
MAX_TASK_TOOL_CALLS = int(
    os.environ.get("REPO_INTELLIGENCE_MAX_TASK_TOOL_CALLS", "12")
)
if MAX_TASK_TOOL_CALLS < 0:
    raise ValueError("REPO_INTELLIGENCE_MAX_TASK_TOOL_CALLS must be >= 0")

# Preserve the repository's whole-run ceiling and add the M20 task-level cap.
M20_ORCHESTRATION_CONFIG = replace(
    M17_ORCHESTRATION_CONFIG,
    max_task_tool_calls=MAX_TASK_TOOL_CALLS,
)

orchestrator = Orchestrator(
    investigator=router,
    analyst=visualizer,
    trace_sink=trace_sink,
    config=M20_ORCHESTRATION_CONFIG,
)

print("Architecture investigator:", type(architect).__name__)
print("Flow investigator:", type(flow_investigator).__name__)
print("Analyst:", type(visualizer).__name__)
print("Run-level tool budget:", M20_ORCHESTRATION_CONFIG.max_total_tool_calls)
print("Per-attempt task tool budget:", M20_ORCHESTRATION_CONFIG.max_task_tool_calls)
print("QueryRouter and M20 Orchestrator initialized.")

## 5. Define LangGraph state

In [ ]:
class AgentAssignments(BaseModel):
    """Readable preview: plan step ID -> concrete specialist slot."""
    agent_assignments: dict[str, str] = Field(default_factory=dict)


class RepoIntelligenceState(TypedDict, total=False):
    objective: str
    plan: InvestigationPlan
    role_assignment: list[RoleAssignment]
    agent_assignments: AgentAssignments
    orchestration_run: Any
    execution_status: str
    execution_error: str

## 6. Planner node

In [ ]:
def planner_node(state: RepoIntelligenceState) -> dict:
    objective = (state.get("objective") or "").strip()
    if not objective:
        raise ValueError("state['objective'] must be a non-empty string")

    request = build_planning_request(
        objective,
        architecture_context=architecture_context,
    )
    response = planner_llm.complete(request)
    plan = plan_from_response(response)

    # Keep deterministic, application-owned plan validation.
    validate_plan(plan)

    print(f"Plan created: {len(plan.steps)} step(s)")
    for step in plan.steps:
        print(f"  {step.id}: {step.goal}")

    return {"plan": plan}

## 7. Role assignment node

In [ ]:
def role_assigner_node(state: RepoIntelligenceState) -> dict:
    plan = state["plan"]
    assignments = assign_roles(plan.steps)

    step_ids = [step.id for step in plan.steps]
    assignment_ids = [item.step_id for item in assignments]

    if len(step_ids) != len(set(step_ids)):
        raise ValueError("The plan contains duplicate step IDs.")
    if len(assignment_ids) != len(set(assignment_ids)):
        raise ValueError("Role assignment contains duplicate step IDs.")
    if set(assignment_ids) != set(step_ids):
        raise ValueError(
            "Role assignments must exactly cover plan step IDs. "
            f"Plan={step_ids}; assignments={assignment_ids}"
        )

    print("Preassigned roles:")
    for item in assignments:
        print(f"  {item.step_id}: {item.role} — {item.reason}")

    return {"role_assignment": assignments}

## 8. Specialist-routing preview node

In [ ]:
def query_router_node(state: RepoIntelligenceState) -> dict:
    """Preview routing without mutating the router used for real execution.

    The repository's QueryRouter has no public dry-run method. Use a fresh
    instance for preview so its memo cannot affect the execution router.
    The actual Orchestrator still invokes `router` during task execution.
    """
    plan = state["plan"]
    role_assignments = state["role_assignment"]
    step_ids = [step.id for step in plan.steps]

    role_by_step: dict[str, RoleAssignment] = {}
    for assignment in role_assignments:
        if assignment.step_id in role_by_step:
            raise ValueError(
                f"Duplicate role assignment for step {assignment.step_id!r}"
            )
        role_by_step[assignment.step_id] = assignment

    if set(role_by_step) != set(step_ids):
        raise ValueError(
            "Role assignments must exactly cover plan step IDs. "
            f"Plan={step_ids}; assignments={list(role_by_step)}"
        )

    preview_router = QueryRouter(
        architecture=architect,
        flow=flow_investigator,
    )
    agent_names: dict[str, str] = {}

    for step in plan.steps:
        role = role_by_step[step.id].role
        if role == "ANALYST":
            agent_name = "visualizer"
        elif role == "INVESTIGATOR":
            specialist = preview_router._resolve(step.id, step.goal)
            if specialist is architect:
                agent_name = "architect"
            elif specialist is flow_investigator:
                agent_name = "flow_investigator"
            else:
                raise ValueError(
                    f"QueryRouter returned unexpected specialist for "
                    f"{step.id!r}: {type(specialist).__name__}"
                )
        else:
            raise ValueError(f"Unsupported role {role!r} for {step.id!r}")

        agent_names[step.id] = agent_name
        print(f"  {step.id}: {role} -> {agent_name}")

    return {
        "agent_assignments": AgentAssignments(agent_assignments=agent_names)
    }

## 9. Orchestrator node — execute through the M20 repository runtime

In [ ]:
def orchestrator_node(state: RepoIntelligenceState) -> dict:
    """Run the validated plan through the repository's existing Orchestrator.

    M20 contracts and task-budget enforcement remain inside the repository
    implementation. This notebook does not duplicate the orchestration loop.
    """
    task = TaskPayload(
        question=state["objective"],
        repository_scope=repository_label,
        commit_scope=current_commit,
    )

    run = orchestrator.execute(
        task,
        state["plan"],
        role_assignments=state["role_assignment"],
    )

    print(f"Orchestration status: {run.status}")
    print(f"Termination reason: {run.termination_reason}")
    print(f"Completed steps: {run.completed_steps}")
    print(f"Failed steps: {run.failed_steps}")
    print(f"Blocked steps: {run.blocked_steps}")
    print(f"Measured tool calls: {run.tool_calls_total}")
    print(f"Configured run tool budget: {M20_ORCHESTRATION_CONFIG.max_total_tool_calls}")
    print(f"Configured per-attempt task budget: {M20_ORCHESTRATION_CONFIG.max_task_tool_calls}")

    return {
        "orchestration_run": run,
        "execution_status": run.status,
        "execution_error": run.termination_reason or "",
    }

## 10. Build and compile the LangGraph workflow

In [ ]:
builder = StateGraph(RepoIntelligenceState)

builder.add_node("planner", planner_node)
builder.add_node("assign_role", role_assigner_node)
builder.add_node("query_router", query_router_node)
builder.add_node("orchestrator", orchestrator_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "assign_role")
builder.add_edge("assign_role", "query_router")
builder.add_edge("query_router", "orchestrator")
builder.add_edge("orchestrator", END)

graph = builder.compile()
display(graph.get_graph().draw_mermaid())

## 11. Invoke the workflow and inspect this run

In [ ]:
objective = "How do agents and MCP servers interact?"
result = graph.invoke({"objective": objective})

print("\n==================== FINAL RUN SUMMARY ====================")
print("Status:", result.get("execution_status"))
print("Termination:", result.get("execution_error"))

run = result.get("orchestration_run")
if run is None:
    raise RuntimeError("LangGraph completed without returning an orchestration run.")

print("Run ID:", run.run_id)
print("Final:", run.final)
print("Completed steps:", run.completed_steps)
print("Failed steps:", run.failed_steps)
print("Blocked steps:", run.blocked_steps)
print("Run tool calls:", run.tool_calls_total)

print("\nTask records:")
for task_record in run.tasks:
    usage = task_record.usage
    usage_total = task_record.usage_total
    print(
        f"  {task_record.task_id}: status={task_record.status}, "
        f"role={task_record.role}, attempts={task_record.attempts}, "
        f"last_attempt_tool_calls={getattr(usage, 'tool_calls', None)}, "
        f"cumulative_tool_calls={getattr(usage_total, 'tool_calls', None)}, "
        f"trace_id={task_record.agent_run_id}"
    )

if run.finding is not None:
    print("\nFinding:")
    display(run.finding)

if run.report is not None:
    print("\nMerged investigation report:")
    display(run.report)

print("\nRole assignments:")
for assignment in result.get("role_assignment", []):
    print(f"  {assignment.step_id}: {assignment.role} — {assignment.reason}")

print("\nConcrete routing preview (not proof of execution):")
print(result.get("agent_assignments"))

## 12. Inspect only traces produced by this orchestration run

In [ ]:
# The Orchestrator records agent_run_id values for attempts belonging to this
# run. Filter by those IDs; do not dump every historical trace in runtime/.
agent_run_ids = {
    item.get("run_id")
    for item in (run.agent_runs or [])
    if item.get("run_id")
}
trace_root = PROJECT_ROOT / "runtime" / "agent_runs"

if not trace_root.exists():
    print(f"Trace directory not found: {trace_root}")
else:
    trace_files = sorted(trace_root.glob("*/trace.json"))
    matching = []

    for trace_path in trace_files:
        try:
            with trace_path.open(encoding="utf-8") as handle:
                trace = json.load(handle)
        except (OSError, json.JSONDecodeError) as exc:
            print(f"Could not read {trace_path}: {exc}")
            continue

        if trace.get("run_id") in agent_run_ids:
            matching.append((trace_path, trace))

    if not matching:
        print("No trace files matched the agent run IDs recorded for this run.")
        print("Recorded agent run IDs:", sorted(agent_run_ids))
    for trace_path, trace in matching:
        summary = trace.get("summary", {})
        print(f"\nTrace file: {trace_path}")
        print("Trace ID:", trace.get("run_id"))
        print("Started:", trace.get("started_at"))
        print("Completed:", trace.get("completed_at"))
        print("Tool calls executed:", summary.get("tool_calls_executed"))
        print("Tool calls blocked:", summary.get("tool_calls_blocked"))
        for event in trace.get("orchestration_events", []):
            print(
                " ",
                event.get("timestamp"),
                event.get("operation"),
                "step=", event.get("step_id"),
                "role=", event.get("agent_role"),
                "status=", event.get("status"),
            )

## 13. How to interpret the result

- `COMPLETED` means the orchestration and final validation accepted the result.
- `INCOMPLETE` can occur even when every step completed: the finding or diagram can explicitly declare that evidence is insufficient. This is an application-quality outcome, not necessarily an orchestration failure.
- The per-attempt budget is active because this notebook constructs `M20_ORCHESTRATION_CONFIG` with `max_task_tool_calls`. The M20 executor forwards the limit to the participant's pre-tool guard.
- `tool_calls_total` counts measured executed calls, not blocked or merely requested calls.
- The graph in this notebook still has one orchestration node that delegates to the repository's sequential Orchestrator. It does not yet dispatch individual tasks as LangGraph nodes, and it does not provide parallel execution. That is intentional for this integration version.